In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [113]:
fin_data_master = pd.read_csv('financial_data.csv') 

In [114]:
prolong_master = pd.read_csv('prolongations.csv')

In [444]:
fin_data = fin_data_master.copy()
display(fin_data.head(3))

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [445]:
prolong = prolong_master.copy()
display(prolong.head(3))

,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич


In [446]:
fin_data = fin_data.drop(['Причина дубля'], axis=1)

In [447]:
end_data = fin_data.copy()

for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x in ['end', 'стоп'] else False)
    
end_data = end_data.groupby(by='id', as_index=False).sum()

for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x == 1 else False)

In [448]:
zero_data = fin_data.copy()

for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 'в ноль' else False)
    
zero_data = zero_data.groupby(by='id', as_index=False).sum()

for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 1 else False)

In [456]:
pay_data = fin_data.copy()

for col in pay_data.columns[1:17]:
    pay_data[col] = pay_data[col].apply(lambda x: np.nan if x in ['end', 'стоп', 'в ноль'] else x)
    pay_data[col] = pay_data[col].fillna('0,00')
    pay_data[col] = pay_data[col].apply(lambda x: x[:-3])
    pay_data[col] = pay_data[col].apply(lambda x: x.replace('\xa0', ''))
    pay_data[col] = pay_data[col].apply(lambda x: x.replace(',', ''))
    pay_data[col] = pay_data[col].astype(int)
    
pay_data = pay_data.groupby(by='id', as_index=False).sum()

def manager_back(txt):
    name_list = txt.split(' ')
    if len(name_list) < 3 :
        print(name_list)
        name = f'{name_list[0]} {name_list[1]}'
    else:    
        t = len(name_list[0])
        name = f'{name_list[0]} {name_list[1]} {name_list[2]}'
        name = name[:-t]
    return name

pay_data['Account'] = pay_data['Account'].apply(manager_back)

['без', 'А/М']
['без', 'А/М']


In [370]:
# prolong МОДИФИКАЦИЯ НАЗВАНИЯ МЕСЯЦА
def first_symbol(txt):
    frst = txt[0]
    frst = frst.upper()
    txt = frst + txt[1:]
    return txt

prolong['month'] = prolong['month'].apply(first_symbol)


In [ ]:
# НЕ ЗАПУСКАТЬ, объединил через JOIN очень коряво
general_table = pay_data.join(
    other = prolong,
    how = 'left',
    on = 'id',
    lsuffix = '_l',
    rsuffix = '_r'
)

In [371]:
general_table = pd.merge(
    left = pay_data,
    right = prolong,
    how = 'left',
    left_on = 'id',
    right_on = 'id'
)

general_table = general_table.drop(['AM'], axis=1)

In [408]:
close_month_data = general_table.copy()

for col in close_month_data.columns[1:17] :
    for string in close_month_data.index :
        if close_month_data.loc[string, 'month'] == col:
            close_month_data.at[string, col] = True
        else: 
            close_month_data.at[string, col] = False
            
close_month_data = close_month_data.groupby(by='id', as_index=False).sum()

for col in close_month_data.columns[1:17]:
    close_month_data[col] = close_month_data[col].apply(lambda x: True if x == 1 else False)
    
#close_month_data = close_month_data.drop(['Account', 'month'], axis=1)

C:\Users\azott\AppData\Local\Temp\ipykernel_1652\885197620.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = False
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\885197620.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = True
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\885197620.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = False
C:\Users\

ПЕРВАЯ ВЕРСИЯ ДО АЛЬТЕРНАТИВЫ, НЕ УВЕРЕН, ЧТО НАДО БЫЛО ДЕЛАТЬ ДЖЕНЕРАЛ

In [265]:
# 1-Я ПРОЛОНГАЦ ИЗ ДЖЕНЕРАЛ
dict_months = {}
for n in range(1, 17):
    dict_months[n] = general_table.columns[n]

first_prolongation = general_table.copy()

first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

for col in range(2, 17) :
    for string in first_prolongation.index :
        if close_month_data.iloc[string, (col - 1)] == False:
            first_prolongation.at[string, dict_months[col]] = 0

In [266]:
# 2-Я ПРОЛОНГАЦ ИЗ ДЖЕНЕРАЛ
second_prolongation = general_table.copy()

second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

for col in range(3, 17) :
    for string in second_prolongation.index :
        if close_month_data.iloc[string, (col - 2)] == False:
            second_prolongation.at[string, dict_months[col]] = 0

In [ ]:
# ЛАСТ-МОНФС ИЗ ДЖЕНЕРАЛ
last_months_data = general_table.copy()

end_progect_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        # по id проекта находим строку с информацией о последних месяцах этого проекта
        end_index = end_data[end_data['id'] == id_project].index
        if (end_data.iloc[end_index, col]).bool() == True :
            end_progect_list.append(id_project)
            last_months_data.iat[string, col] = 0
            continue
        close_month_index = close_month_data[close_month_data['id'] == id_project].index
        if (close_month_data.iloc[close_month_index, col]).bool() == False :
            last_months_data.iat[string, col] = 0
        else:
            zero_index = zero_data[zero_data['id'] == id_project].index
            if (zero_data.iloc[zero_index, col]).bool() == True :
                pay_index = pay_data[pay_data['id'] == id_project].index
                last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]

C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:11: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (end_data.iloc[end_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:16: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (close_month_data.iloc[close_month_index, col]).bool() == False :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:20: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (zero_data.iloc[zero_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2853228903.py:22: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]


АЛЬТЕРНАТИВА

In [400]:
# 1-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]

first_prolongation = pay_data.copy()

first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

for col in range(2, 17) :
    for string in first_prolongation.index :
        if close_month_data.iloc[string, (col - 1)] == False:
            first_prolongation.at[string, dict_months[col]] = 0
            
first_prolongation = first_prolongation.groupby(by='Account', as_index=False).sum()

first_prolongation = first_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
first_prolongation = first_prolongation.drop([0], axis=0)

In [457]:
first_cnt_project = close_month_data.copy()

first_cnt_project['Account'] = first_cnt_project['Account'].apply(manager_back)
first_cnt_project['Account'].value_counts()

['']


IndexError: list index out of range

In [422]:
# 1-ОЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
first_cnt_project = close_month_data.copy()
first_cnt_project['Account'] = first_cnt_project['Account'].apply(manager_back)
first_cnt_project = first_cnt_project.drop(['id', 'month'], axis=1)
first_cnt_project = first_cnt_project.groupby(by='Account', as_index=False).sum()

In [423]:
first_cnt_project

,Account,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024
0,,20,55,11,20,23,19,18,23,17,18,29,19,29,58,0,0
1,Васильев Артем АлександровичВасильев Артем Але...,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,Кузнецов Михаил ИвановичКузнецов Михаил Иванович,0,1,2,1,0,0,1,0,0,1,0,0,1,1,0,0
3,Смирнова Ольга ВладимировнаСмирнова Ольга Влад...,0,1,0,0,0,1,0,0,2,0,0,2,1,1,0,0
4,Соколова Анастасия Викторовна,4,9,6,7,4,6,3,6,4,6,8,2,7,8,0,0
5,Соколова Анастасия ВикторовнаСоколова Анастаси...,0,0,0,0,2,0,0,2,0,1,3,1,2,1,0,0


In [401]:
# 2-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА
second_prolongation = pay_data.copy()

second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

for col in range(3, 17) :
    for string in second_prolongation.index :
        if close_month_data.iloc[string, (col - 2)] == False:
            second_prolongation.at[string, dict_months[col]] = 0

second_prolongation = second_prolongation.groupby(by='Account', as_index=False).sum()

second_prolongation = second_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
second_prolongation = second_prolongation.drop([0], axis=0)

In [375]:
# ЛАСТ-МОНФС ИЗ ПЭЙ-ДАТА
last_months_data = pay_data.copy()

end_project_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        if id_project in end_project_list :
            last_months_data.iat[string, col] = 0
        else:
            # по id проекта находим строку с информацией о последних месяцах этого проекта
            end_index = end_data[end_data['id'] == id_project].index
            if (end_data.iloc[end_index, col]).bool() == True :
                end_project_list.append(id_project)
                last_months_data.iat[string, col] = 0
                continue
            close_month_index = close_month_data[close_month_data['id'] == id_project].index
            if (close_month_data.iloc[close_month_index, col]).bool() == False :
                last_months_data.iat[string, col] = 0
            else:
                zero_index = zero_data[zero_data['id'] == id_project].index
                if (zero_data.iloc[zero_index, col]).bool() == True :
                    pay_index = pay_data[pay_data['id'] == id_project].index
                    last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]

C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2073179320.py:15: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (end_data.iloc[end_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2073179320.py:20: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (close_month_data.iloc[close_month_index, col]).bool() == False :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2073179320.py:24: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (zero_data.iloc[zero_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\2073179320.py:26: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]


In [376]:
coef_sum_dict = {}

for last_month in range(2, 14) :
    goal_month = last_months_data.columns[last_month + 1]
    coef_sum = last_months_data[last_months_data.columns[last_month]].sum()
    #print(f'целевой месяц {goal_month}, а сумма будет из месяца {last_month} и равна {coef_sum}')
    coef_sum_dict[goal_month] = coef_sum

In [391]:
first_coef = first_prolongation.copy()

for col in first_coef.columns[1:13] :
    for string in first_coef.index:
        first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_1652\708089269.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.186' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\708089269.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.257' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\708089269.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.247' has dtype incompatible with int64, plea

In [392]:
second_coef = second_prolongation.copy()

for col in second_coef.columns[1:13] :
    for string in second_coef.index:
        second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_1652\3344493090.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.059' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\3344493090.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.386' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_1652\3344493090.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.533' has dtype incompatible with int6

In [404]:
last_months_data.head(3)

,id,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,15,0,439280,0,102433,102433,138158,0,102433,0,0,0,0,0,0,0,0,Иванова Мария Сергеевна
1,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Иванова Мария Сергеевна
2,31,0,55100,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Васильев Артем Александрович


In [ ]:
# НЕ ЗАПУСКАТЬ, проверил на пропуски
for elem in general_table['month']:
    if elem is np.nan:
        print('есть')

есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
